## AE33 TCP/IP Communication Guide

This notebook demonstrates how to communicate with the **AE33 Aethalometer** using its TCP/IP interface for real-time data acquisition and control.

**License:** Aerosol Magee Scientific Software License. See LICENSE file for full terms.

In [1]:
from io import StringIO
import re

import pandas as pd

from aerosol_magee_pytools.data_access.tcp_ip import request_tcp

In [2]:
# IP of the instrument - change it to the actual IP address of your AE33 instrument
instrument_ip = '10.10.10.224'

timeout = 2.0  # seconds

In [3]:
# default column names
COLUMNS_AE33_DATA = ['SerialNumber', 'ID', 'StartTime', 'EndTime', 'SetupID', 'SetupTimestamp',
                     'Ref1', 'Sens11', 'Sens12', 'Ref2', 'Sens21', 'Sens22',
                     'Ref3', 'Sens31', 'Sens32', 'Ref4', 'Sens41', 'Sens42',
                     'Ref5', 'Sens51', 'Sens52', 'Ref6', 'Sens61', 'Sens62',
                     'Ref7', 'Sens71', 'Sens72',
                     'BC11', 'BC12', 'BC1', 'BC21', 'BC22', 'BC2',
                     'BC31', 'BC32', 'BC3', 'BC41', 'BC42', 'BC4',
                     'BC51', 'BC52', 'BC5', 'BC61', 'BC62', 'BC6',
                     'BC71', 'BC72', 'BC7',
                     'K1', 'K2', 'K3', 'K4', 'K5', 'K6', 'K7', 'BB',
                     'Pressure', 'Temp', 'Flow1', 'Flow2', 'FlowC',
                     'T_controller', 'T_supply', 'T_LED', 'ControllerStatus',
                     'LEDStatus', 'DetectorStatus', 'ValveStatus', 'Status',
                     'TapeAdvanceCount', 'TapeAdvanceLeft', 'CPU', 'DiskSpace', 'NumConnections']
COLUMNS_AE33_EXTERNAL_DEVICE_DATA = ['SerialNumber', 'ID', 'DataID', 'DeviceID', 'DeviceData']
COLUMNS_AE33_SETUP = ['serial', 'ID', 'SerialNumber', 'Timestamp', 'FirmwareVer', 'SoftwareVer',
                      'DataCenterIP', 'AutoConnect', 'InletFilter',
                      'Timebase', 'SG1', 'SG2', 'SG3', 'SG4', 'SG5', 'SG6', 'SG7',
                      'C', 'Area', 'Zeta', 'Aff', 'Abb', 'Pressure', 'Temp',
                      'ATNf1', 'ATNf2', 'Kmax', 'Kmin', 'Flow', 'FlowRepStd',
                      'PumpPresetValue', 'FlowFormulaA0', 'FlowFormulaA1',
                      'FlowFormulaB0', 'FlowFormulaB1', 'FlowFormulaC0',
                      'FlowFormulaC1', 'FlowFormulaD', 'FlowFormulaE',
                      'FlowFormulaF', 'TAtype', 'TAatnMax', 'TAinterval',
                      'TAtime', 'TapeRightFormulaK', 'TapeRightFormulaN',
                      'TapeLeftFormulaK', 'TapeLeftFormulaN', 'WarmUpInterval',
                      'AutoTestEnabled', 'AutoTestType', 'AutoTestDay', 'AutoTestTime',
                      'MeasureTimeStamp', 'HomeInfo', 'Display', 'About', 'DST', 'TimeZone',
                      'TapeAdvanceAdjust', 'ExternalID', 'BHparamID', 'TimeSync', 'DHCP',
                      'InstrumentIP', 'SubnetMask', 'Gateway', 'Baud', 'NTPserver']

In [4]:
### command HELLO
command_ae33 = 'HELLO\r\n'

received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33,
                            timeout=timeout)
print(received_text)

AE33>Aethalometer information
Server type: AE33
Instrument serialnumber: AE33-S06-00565
Time: 23/Jun/2026 09:04:47
Database version: 1.7.3
Connected Clients: 3
ID: 1441398  IP: 192.168.45.37:60073  Time: 23 Jun 2026 09:04:46  Streaming: False
ID: 1384580  IP: 10.10.10.253:49760  Time: 16 Jun 2026 05:12:28  Streaming: True
ID: 1384579  IP: 10.10.10.253:49751  Time: 16 Jun 2026 05:12:28  Streaming: True

AE33>


In [5]:
### command MAXID Data
command_ae33 = 'MAXID Data\r\n'

received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33,
                            timeout=timeout)
print(received_text)

# # parse data
max_data_id = int(re.search(r'AE33>(\d+)', received_text).group(1))
print()
print(max_data_id)


AE33>3176881
AE33>

3176881


In [6]:
### command FETCH Data - collect last 5 rows from table Data
command_ae33 = f'FETCH Data {max_data_id-5} {max_data_id}\r\n'

received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33,
                            timeout=timeout)
print(received_text)

# in the end, you can parse data, for example convert it into pandas dataframe
df_ae33_data = pd.read_csv(StringIO(received_text.replace('AE33>', '').strip()),
                           sep='|',
                           names=COLUMNS_AE33_DATA)
print()
print(df_ae33_data)

AE33>AE33>AE33-S06-00565|3176876|6/23/2026 8:58:00 AM|6/23/2026 8:59:00 AM|26|7/29/2025 7:24:13 AM|905532|308215|500733|876454|365612|545013|891851|409699|557817|891636|430687|563887|899041|534721|692550|780262|656950|810140|859714|681640|815270|5|20|5|3|23|3|4|-30|3|-8|18|-7|32|135|31|38|5|39|35|113|37|-8.958477E-05|-0.00223335|-0.001961019|-0.0007833776|-0.0008777904|0.001229791|0.002427158|0|101325.0|21.1|3222|1178|4400|34.0|38.0|36.0|0|10|10|0|4|6066|285|6|5234|2
AE33>AE33-S06-00565|3176877|6/23/2026 8:59:00 AM|6/23/2026 9:00:00 AM|26|7/29/2025 7:24:13 AM|905555|308191|500716|876527|365619|545027|891913|409699|557836|891693|430704|563906|899105|534747|692589|780317|656969|810174|859789|681676|815313|99|156|98|78|192|69|94|131|85|37|126|36|39|70|37|95|178|98|85|233|91|-8.918323E-05|-0.002229017|-0.001961183|-0.0007795616|-0.0008780891|0.001233478|0.002436896|0|101325.0|21.1|3222|1177|4399|34.0|40.0|36.0|0|10|10|0|4|6066|285|6|5234|2
AE33>AE33-S06-00565|3176878|6/23/2026 9:00:00 AM|6

In [7]:
### collect data from Setup table
command_ae33 = f'FETCH SETUP\r\n'

received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33,
                            timeout=timeout)
print(received_text)

# in the end, you can parse data, for example convert it into pandas dataframe
df_ae33_setup = pd.read_csv(StringIO(received_text.replace('AE33>', '').strip()),
                            sep='|',
                            names=COLUMNS_AE33_SETUP)
# drop duplicated serial column from setup data
df_ae33_setup = df_ae33_setup.drop(columns=['serial'], errors='ignore')
print()
print(df_ae33_setup)

AE33>AE33-S06-00565|1|AE33-S06-00565|10/30/2020 4:47:54 PM|533|1.4.9.1|10.10.10.253:8007|1|0|60|18.47|14.54|13.14|11.58|10.35|7.77|7.19|1.39|0.785|0.02|1|2|101325|25.00|10|30|0.015|-0.005|2000|1|585|-1952.80444335937|-2505.5458984375|11.6349792480469|12.9688386917114|0.000753700849600136|-0.000350014335708693|166.16487121582|0.0840134471654892|-4.06122467211389E-07|1|120|12|3/13/2017 10:57:40 AM|1.04204201698303|1.88889074325562|1.12658226490021|-45.7848091125488|1|1|0|2|1/1/2014 12:00:00 AM|1|0|1|0|0|Coordinated Universal Time|10|1|1|1|1|192.168.0.2|255.255.255.0|192.168.0.1|115200|pool.ntp.org
AE33>AE33>AE33-S06-00565|13|AE33-S06-00565|9/14/2021 7:23:04 AM|540|1.5.0.1|10.10.10.253:8007|1|0|60|18.47|14.54|13.14|11.58|10.35|7.77|7.19|1.39|0.785|0.02|1|2|101325|25.00|10|30|0.015|-0.005|2000|1|585|-1952.80444335937|-2505.5458984375|11.6349792480469|12.9688386917114|0.000753700849600136|-0.000350014335708693|166.16487121582|0.0840134471654892|-4.06122467211389E-07|1|120|12|3/13/2017 10:57

In [8]:
### collect data from ExtDeviceData table - get MAXID and then fetch last 5 rows

command_ae33 = 'MAXID ExtDeviceData\r\n'
received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33,
                            timeout=timeout)
print(received_text)
max_ext_device_data_id = int(re.search(r'AE33>(\d+)', received_text).group(1))

command_ae33 = f'FETCH ExtDeviceData {max_ext_device_data_id-5} {max_ext_device_data_id}\r\n'
received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33,
                            timeout=timeout)
print(received_text)

# in the end, you can parse data, for example convert it into pandas dataframe
df_ae33_device_data = pd.read_csv(StringIO(received_text.replace('AE33>', '').strip()),
                           sep='|',
                           names=COLUMNS_AE33_EXTERNAL_DEVICE_DATA)
print()
print(df_ae33_device_data)

AE33>4460321
AE33>
AE33>AE33-S06-00565|4460316|3176876|15|99
AE33>AE33>AE33-S06-00565|4460317|3176877|15|99
AE33>AE33-S06-00565|4460318|3176878|15|99
AE33>AE33-S06-00565|4460319|3176879|15|99
AE33>AE33-S06-00565|4460320|3176880|15|99
AE33>AE33-S06-00565|4460321|3176881|15|99
AE33>

     SerialNumber       ID   DataID  DeviceID  DeviceData
0  AE33-S06-00565  4460316  3176876        15          99
1  AE33-S06-00565  4460317  3176877        15          99
2  AE33-S06-00565  4460318  3176878        15          99
3  AE33-S06-00565  4460319  3176879        15          99
4  AE33-S06-00565  4460320  3176880        15          99
5  AE33-S06-00565  4460321  3176881        15          99
